# 01 — Datasets

The Background figures of the paper: class distributions of the simulated TF and the
real RBP data (Fig. 1), the single-cell UMAP (Fig. 2) and the single-cell class
distribution (Fig. 3).

In [ ]:
import os, sys, pathlib

# repo root, whether the notebook is run from notebooks/ or from the root
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

# released data: the summary bundle (all notebooks) and the raw per-pass predictions
# (notebooks 04 and 06). Set the variables before starting Jupyter to use other locations.
os.environ.setdefault('UQFIG_BUNDLE', str(ROOT / 'figure_data_bundle'))
os.environ.setdefault('UQFIG_PREDS', str(ROOT / 'cluster_preds'))
OUT = ROOT / 'out'; OUT.mkdir(exist_ok=True)

from uq_genomics.figures import style as S
S.apply_rc()
print('bundle :', os.environ['UQFIG_BUNDLE'])
print('preds  :', os.environ['UQFIG_PREDS'])
print('output :', OUT)

## Class distributions (Fig. 1, Fig. 3)

In [ ]:
from uq_genomics.figures import plots_datasets as PD
res = PD.run(str(OUT))
for r in (res or []):
    print(f"{r.get('id',''):40s} {r.get('status',''):8s} {r.get('reason','')}")

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'dataset_classdist_tf_bal.png'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=360)) if p.exists() else None
p = OUT / 'dataset_classdist_rbp_bal.png'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=360)) if p.exists() else None
p = OUT / 'dataset_classdist_gex_bal.png'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=360)) if p.exists() else None

## Single-cell UMAP (Fig. 2)

Panel a is the dataset's own UMAP of all 69,249 cells; panel b the UMAP of the 20-D
Liam embedding of the 28,214 cells of the four selected types. Both coordinate sets
ship with the data bundle (`_full_xumap_labels.pkl`, `umap_liam_4_cell_types_coords.pkl`).

In [ ]:
from uq_genomics.figures import plots_umap as PU
PU.build(str(OUT))

In [ ]:
from IPython.display import IFrame, display
p = OUT / 'UMAP_liam_embedding_v2.pdf'
print(p, '->', 'exists' if p.exists() else 'MISSING')
display(IFrame(str(p), width=900, height=420)) if p.exists() else None

## Training-set class sizes

The two real use cases construct imbalance differently, which matters for how the
sensitivity results in notebook 04 are read. The single-cell setting depletes its
minority class (Erythroblast, 2673 → 587 cells); the RBP setting leaves its smallest
class almost unchanged and enriches the majority classes instead.

In [ ]:
import numpy as np
from uq_genomics.figures import data_access as DA
cd = DA.bundle_part('classdist')
for task, kb, ki, names in [
    ('RBP', ('rbp',(('balanced',True),('multi_label',False)),7),
            ('rbp',(('balanced',False),('multi_label',False)),7),
            ['MBNL1','PUM2','QKI']),
    ('GEX', ('gex',(('balanced',True),),7), ('gex',(('balanced',False),),7),
            ['CD14+ Mono','CD4+ T activated','Erythroblast','NK'])]:
    b = np.asarray(cd[kb]['y_train']); i = np.asarray(cd[ki]['y_train'])
    print(f'--- {task}: balanced -> imbalanced training set ---')
    for n, x, y in zip(names, b, i):
        print(f'  {n:18s} {x:6d} -> {y:6d}   x{y/x:5.2f}')